# 06. Pipeline de Reentrenamiento - Human in the Loop (HITL)

Este notebook contiene el pipeline interactivo para llevar a cabo el **reentrenamiento y fine-tuning** del modelo de detección de anomalías viales, cerrando el ciclo de **Human in the Loop (HITL)**.

A diferencia del resto de la arquitectura que corre en Docker y Kubernetes (API, Workers y Frontend), este proceso de entrenamiento se realiza en un cuaderno interactivo aislado. A continuación, justificamos esta decisión de diseño antes de ejecutar el código.

## 🏛️ Decisiones de Diseño y MLOps: ¿Por qué desacoplar el entrenamiento del backend?

Durante el diseño del sistema, evaluamos si el reentrenamiento debía activarse con un botón desde la página web (corriendo dentro del Docker de producción). Decidimos mantenerlo **desacoplado y en una notebook** por las siguientes razones de ingeniería:

1. **Saturación de Cómputo (Consumo de CPU/GPU):** El entrenamiento de redes neuronales convolucionales como YOLO consume el 100% de la capacidad de procesamiento del hardware. Si este proceso corriera dentro de los mismos servidores de producción que la API, provocaría caídas del servicio, lentitud y timeouts a los usuarios que estén cargando videos en ese momento.
2. **Asincronía y Uso Escaso (Batch):** El reentrenamiento no ocurre en tiempo real. Es un proceso de tipo "lote" (Batch) que se ejecuta esporádicamente (por ejemplo, una vez al mes o cuando se acumulan suficientes auditorías).
3. **Complejidad del Pipeline (Overengineering):** Integrar el reentrenamiento en la interfaz web requeriría programar un Model Registry (ej. MLflow), un visor de curvas de pérdida/precisión, lógica para almacenar múltiples archivos `.pt` históricos y un sistema de control de versiones. Esto agregaría complejidad excesiva en etapas iniciales.
4. **Portabilidad a la Nube (Google Colab):** Al estar diseñado en este notebook, podés mover los datos en segundos a entornos optimizados para inteligencia artificial con **GPU gratuitas** (como Google Colab o RunPod) sin necesidad de configurar drivers de Nvidia, CUDA o compilar C++ de forma local.

## 🧠 ¿Cómo evitar el "Olvido Catastrófico"?

Si reentrenamos el modelo **únicamente** con las imágenes nuevas obtenidas de la auditoría web (que suelen ser pocas, ej. 50 o 100 imágenes), el modelo sufrirá de **Olvido Catastrófico** (*Catastrophic Forgetting*). Es decir, optimizará sus pesos matemáticos para desempeñarse perfectamente en esas 100 imágenes nuevas y empezará a fallar en la detección del resto de los baches que antes detectaba bien.

Para solucionar esto, este notebook implementa dos estrategias combinadas:
* **Dataset Mixto (Replay Data):** Mezclamos las imágenes nuevas auditadas con una muestra representativa (el 10% o 20%) del dataset base original con el que se entrenó la versión activa.
* **Congelamiento de Capas (Layer Freezing):** Congelamos el *Backbone* de YOLO (las primeras 10 capas). Esto asegura que el modelo no modifique su habilidad básica para detectar texturas de asfalto y formas, concentrándose únicamente en adaptar la clasificación del daño en el *Head* del modelo.

## 1. Instalación de Librerías y Carga de Entorno

Instalamos los paquetes necesarios para conectarnos a la base de datos PostgreSQL, descargar las imágenes de MinIO y ejecutar YOLO con la librería Ultralytics.

In [ ]:
%pip install minio sqlalchemy psycopg2-binary opencv-python pillow ultralytics python-dotenv gdown

In [ ]:
import os
import sys
import json
import random
import shutil
from pathlib import Path
from minio import Minio
from PIL import Image
from sqlalchemy import create_engine, text

## 2. Configurar Conexiones e Inicializar Clientes (Google Cloud / Producción)

Copia el archivo de ejemplo y completa los valores necesarios en tu nuevo `.env`:
```bash
cp .env.example .env
```


In [ ]:
from dotenv import load_dotenv
load_dotenv()  # Carga variables desde un archivo .env si existe

# Credenciales de PostgreSQL (se buscan en el entorno o archivo .env)
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_USER = os.getenv("DB_USER", "postgres")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")
DB_NAME = os.getenv("DB_NAME", "baches_db")

# Credenciales y Endpoint de MinIO
MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_USER = os.getenv("MINIO_USER", "minioadmin")
MINIO_PASSWORD = os.getenv("MINIO_PASSWORD", "minioadmin")

# ==============================================================================
# Conexión automática
# ==============================================================================
DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

print(f"DATABASE_URL configurada: postgresql://{DB_USER}:****@{DB_HOST}:{DB_PORT}/{DB_NAME}")
print(f"MinIO Endpoint configurado: {MINIO_ENDPOINT}")

In [ ]:
try:
    engine = create_engine(DATABASE_URL)
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))
    print("Conexión exitosa a PostgreSQL.")
except Exception as e:
    print(f"Error al conectar a PostgreSQL: {e}")

try:
    minio_client = Minio(
        MINIO_ENDPOINT,
        access_key=MINIO_USER,
        secret_key=MINIO_PASSWORD,
        secure=False
    )
    minio_client.list_buckets()
    print("Conexión exitosa a MinIO.")
except Exception as e:
    print(f"Error al conectar a MinIO: {e}")

## 3. Extracción de Datos Auditados (HITL)

Consultamos los registros cuyo estado de auditoría sea `'verificado'` (baches reales aprobados por el operador) o `'falso_positivo'` (detecciones erróneas descartadas). Las imágenes marcadas como falsos positivos se guardarán en el dataset como **backgrounds** (etiquetas vacías) para reducir la tasa de falsos positivos en el modelo reentrenado.

In [ ]:
query = text("""
    SELECT id, video_id, tipo_dano, frame_minio_path, bbox, estado_auditoria 
    FROM deteccion 
    WHERE estado_auditoria IN ('verificado', 'falso_positivo')
""")

with engine.connect() as conn:
    resultados = conn.execute(query).fetchall()

print(f"Detecciones auditadas encontradas: {len(resultados)}")

## 4. Crear Dataset y Descargar Imágenes

Descargamos las imágenes desde los buckets de MinIO (`detecciones` para aprobados y `backgrounds-reentrenamiento` para falsos positivos) y convertimos los bboxes de la BD a formato YOLO normalizado.

In [ ]:
DIR_OUTPUT = Path("dataset_reentrenamiento")
DIR_IMAGES_TRAIN = DIR_OUTPUT / "images" / "train"
DIR_IMAGES_VAL = DIR_OUTPUT / "images" / "val"
DIR_LABELS_TRAIN = DIR_OUTPUT / "labels" / "train"
DIR_LABELS_VAL = DIR_OUTPUT / "labels" / "val"

for folder in [DIR_IMAGES_TRAIN, DIR_IMAGES_VAL, DIR_LABELS_TRAIN, DIR_LABELS_VAL]:
    folder.mkdir(parents=True, exist_ok=True)

# Mapeo estático y estricto del proyecto (idéntico al entrenamiento base)
clases_map = {"D20": 0, "D40": 1, "calle_tierra": 2}
print(f"Mapeo de clases estático: {clases_map}")

tmp_download_dir = Path("tmp_downloads")
tmp_download_dir.mkdir(exist_ok=True)
dataset_items = []

for row in resultados:
    det_id = row.id
    minio_path = row.frame_minio_path
    bbox = row.bbox
    estado = row.estado_auditoria
    tipo_dano = row.tipo_dano

    if not minio_path:
        continue

    ext = Path(minio_path).suffix or ".jpg"
    img_name = f"det_{det_id}_{estado}{ext}"
    label_name = f"det_{det_id}_{estado}.txt"
    local_img_path = tmp_download_dir / img_name
    local_label_path = tmp_download_dir / label_name

    try:
        # Descargar según el bucket donde se guardó en la auditoría
        if estado == "verificado":
            minio_client.fget_object("detecciones", minio_path, str(local_img_path))
        elif estado == "falso_positivo":
            minio_client.fget_object("backgrounds-reentrenamiento", minio_path, str(local_img_path))

        with Image.open(local_img_path) as img:
            img_width, img_height = img.size

        lineas_anotacion = []
        if estado == "verificado" and bbox:
            if isinstance(bbox, str):
                bbox = json.loads(bbox)
            
            x1, y1 = float(bbox["x1"]), float(bbox["y1"])
            x2, y2 = float(bbox["x2"]), float(bbox["y2"])
            
            w_box = x2 - x1
            h_box = y2 - y1
            x_center = x1 + (w_box / 2.0)
            y_center = y1 + (h_box / 2.0)

            x_center_norm = max(0.0, min(1.0, x_center / img_width))
            y_center_norm = max(0.0, min(1.0, y_center / img_height))
            w_norm = max(0.0, min(1.0, w_box / img_width))
            h_norm = max(0.0, min(1.0, h_box / img_height))

            class_id = clases_map.get(tipo_dano)
            if class_id is not None:
                lineas_anotacion.append(f"{class_id} {x_center_norm:.6f} {y_center_norm:.6f} {w_norm:.6f} {h_norm:.6f}")

        # El archivo de etiqueta quedará vacío si es falso positivo (imagen background)
        with open(local_label_path, "w") as f_lbl:
            f_lbl.write("\n".join(lineas_anotacion))

        dataset_items.append({
            "img_path": local_img_path,
            "label_path": local_label_path,
            "img_name": img_name,
            "label_name": label_name
        })
    except Exception as e:
        print(f"Error descargando detección ID {det_id}: {e}")

## 5. Integración con el Dataset Original de Moreno (Evitar Olvido)

Para mezclar las nuevas auditorías con las imágenes originales del entrenamiento base y mitigar el **olvido catastrófico**, el notebook incluye una celda especial que descarga el dataset original desde un enlace compartido público de Google Drive usando `gdown`.

In [ ]:
# Descargar una muestra del Dataset Base para mezclarlo (Replay Data)
ID_DRIVE_DATASET_BASE = "1t2k5_rADlHczpZWwmvewc2pNdZBFs21v"  # Enlace público de Google Drive
ruta_base_zip = Path("dataset_base_original.zip")

if not Path("dataset_base_original").exists():
    print("Descargando dataset base original de Moreno desde Google Drive...")
    !gdown --id {ID_DRIVE_DATASET_BASE} -O {ruta_base_zip}
    if ruta_base_zip.exists():
        shutil.unpack_archive(str(ruta_base_zip), "dataset_base_original")
        print("Dataset base original descargado y descomprimido.")
        os.remove(ruta_base_zip)
else:
    print("Dataset base original ya se encuentra disponible localmente.")

In [ ]:
# Mezclamos los datos nuevos auditados con una porción de imágenes originales
ruta_original_imagenes = Path("dataset_base_original/train/images")
ruta_original_etiquetas = Path("dataset_base_original/train/labels")

muestras_adicionales = 0
if ruta_original_imagenes.exists():
    archivos_orig = list(ruta_original_imagenes.glob("*.jpg"))
    cant_muestra = min(len(archivos_orig), 100)
    muestra_original = random.sample(archivos_orig, cant_muestra)

    for img_path in muestra_original:
        lbl_name = img_path.stem + ".txt"
        lbl_path = ruta_original_etiquetas / lbl_name
        if lbl_path.exists():
            shutil.copy(str(img_path), str(tmp_download_dir / img_path.name))
            shutil.copy(str(lbl_path), str(tmp_download_dir / lbl_name))
            dataset_items.append({
                "img_path": tmp_download_dir / img_path.name,
                "label_path": tmp_download_dir / lbl_name,
                "img_name": f"orig_{img_path.name}",
                "label_name": f"orig_{lbl_name}"
            })
            muestras_adicionales += 1

print(f"Se inyectaron {muestras_adicionales} imágenes del dataset original para mitigar el olvido catastrófico.")

## 6. Realizar el Split Train / Validation y Generar YAML

Dividimos el dataset mezclado reservando el 80% para entrenamiento y el 20% para validación.

In [ ]:
random.seed(42)
random.shuffle(dataset_items)

split_idx = int(len(dataset_items) * 0.8)
train_items = dataset_items[:split_idx]
val_items = dataset_items[split_idx:]

def mover_elementos(items, img_dir, lbl_dir):
    for item in items:
        shutil.move(str(item["img_path"]), str(img_dir / item["img_name"]))
        shutil.move(str(item["label_path"]), str(lbl_dir / item["label_name"]))

mover_elementos(train_items, DIR_IMAGES_TRAIN, DIR_LABELS_TRAIN)
mover_elementos(val_items, DIR_IMAGES_VAL, DIR_LABELS_VAL)

# Limpiar descargas temporales
if tmp_download_dir.exists():
    shutil.rmtree(tmp_download_dir)

print(f"Dataset final estructurado en '{DIR_OUTPUT}':")
print(f"  - Train: {len(train_items)} elementos")
print(f"  - Val: {len(val_items)} elementos")

In [ ]:
# Crear el dataset.yaml
yaml_content = f"""
path: {DIR_OUTPUT.resolve().as_posix()}
train: images/train
val: images/val

names:
    0: D20
    1: D40
    2: calle_tierra
"""

yaml_file = DIR_OUTPUT / "dataset.yaml"
with open(yaml_file, "w") as f_yaml:
    f_yaml.write(yaml_content.strip())

print(f"Archivo YAML de YOLO creado en: {yaml_file.resolve()}")

## 7. Ejecutar Fine-Tuning de YOLO

Cargamos los pesos actuales de producción (`best.pt`) y ejecutamos el entrenamiento utilizando la librería `ultralytics`. 

**Configuración anti-olvido:**
* `freeze=10`: Congela el Backbone (primeras 10 capas).
* `lr0=0.001`: Tasa de aprendizaje inicial muy baja para evitar descalibrar el conocimiento previo.

In [ ]:
from ultralytics import YOLO
import os
from pathlib import Path

# Buscamos si existe un progreso previo del reentrenamiento para reanudarlo
ultimo_peso = os.path.join("runs_hitl", "reentrenamiento_vial", "weights", "last.pt")

if os.path.exists(ultimo_peso):
    print(f"⚠️ Reentrenamiento previo detectado. Reanudando desde: {ultimo_peso}")
    model = YOLO(ultimo_peso)
    model.train(
        resume=True,
        data=str(yaml_file.resolve())
    )
else:
    # Si no hay entrenamiento interrumpido, buscamos el modelo base para el fine-tuning
    ruta_modelo_actual = Path("best.pt")
    modelo_base = "best.pt" if ruta_modelo_actual.exists() else "yolo26s.pt"

    print(f"🚀 Iniciando nuevo reentrenamiento desde: {modelo_base}")
    if not ruta_modelo_actual.exists():
        print(f"Aviso: No se encontró 'best.pt' en el directorio actual. Se utilizará el modelo base'{modelo_base}'.")

    model = YOLO(modelo_base)

    # Iniciar entrenamiento con parámetros específicos de fine-tuning (HITL)
    results = model.train(
        data=str(yaml_file.resolve()),
        epochs=15,             # 15 épocas son suficientes para afinar los pesos
        imgsz=640,
        batch=8,
        device="0" if os.environ.get("CUDA_VISIBLE_DEVICES") else "cpu",  # GPU si está disponible
        freeze=10,             # Congela las primeras 10 capas (Backbone)
        lr0=0.001,             # Tasa de aprendizaje baja conservadora
        project="runs_hitl",
        name="reentrenamiento_vial",
        exist_ok=True,
        # Hiperparámetros de aumento de datos
        hsv_s=0.5,
        hsv_v=0.5,
        degrees=5.0,
        fliplr=0.5,
        flipud=0.0,            # Desactivado para mantener orientación del asfalto
        mosaic=0.5             # Reducido para que los baches lejanos no se vuelvan indetectables
    )

print("\n✅ Proceso de reentrenamiento finalizado.")

## 8. Despliegue en Producción

Una vez validado que las métricas de precisión mejoraron en el set de validación:
1. Copiá el nuevo archivo `best.pt` obtenido en `runs_hitl/reentrenamiento_vial/weights/best.pt`.
2. Reemplazá el archivo existente en tu worker de inferencia activa: `pics_proyecto/worker/best.pt`.
3. Reiniciá el contenedor de inferencia para que cargue los nuevos pesos en memoria:
   ```bash
   docker compose restart worker_procesamiento
   ```